# ⚙️ Vigyan-1.5B Engineering: Electrical, Mechanical & Systems

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shreyansh001boy-tech/vigyan-7b-demo/blob/main/qwen_stem_demos/1_5b_engineering_colab.ipynb)
&nbsp;[![License: CC BY-NC 4.0](https://img.shields.io/badge/License-CC_BY--NC_4.0-lightgrey.svg)](https://creativecommons.org/licenses/by-nc/4.0/)
&nbsp;[![Hugging Face Adapter](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-Adapter-yellow)](https://huggingface.co/shreyansh12183/vigyan-1.5b-adapter-engineering)

---

### 🌟 Model Overview
Specialized 1.5B Small Language Model (SLM) calibrated for **Core Engineering & Circuits**.
- **Base Model:** `deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B`
- **Domain Adapter:** `shreyansh12183/vigyan-1.5b-adapter-engineering` (100% Public)
- **Reasoning Engine:** Emits step-by-step `<think> ... </think>` scratchpad reasoning tokens before answering.
- **Hardware Requirement:** Free Google Colab **Tesla T4 GPU** (Allocates in **~1.5 GB VRAM** in 4-bit NF4).
- **License:** CC BY-NC 4.0 (Free for academic, student, and non-commercial research use).

In [ ]:
!pip install -q transformers peft accelerate bitsandbytes gradio
print("✅ Runtime packages installed.")

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

BASE_ID = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
ADAPTER_ID = "shreyansh12183/vigyan-1.5b-adapter-engineering"

print(f"⚡ PyTorch: {torch.__version__} | CUDA: {torch.cuda.get_device_name(0)}")
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
tokenizer = AutoTokenizer.from_pretrained(BASE_ID, trust_remote_code=True)
base = AutoModelForCausalLM.from_pretrained(BASE_ID, quantization_config=bnb_config, device_map="auto", trust_remote_code=True)
model = PeftModel.from_pretrained(base, ADAPTER_ID).eval()
print("✓ Vigyan-1.5B Model loaded in VRAM (~1.5 GB)!")

In [ ]:
import gradio as gr

def solve(query, history):
    prompt = f"<|im_start|>user\n{query}\nPlease reason step by step inside <think> tags.<|im_end|>\n<|im_start|>assistant\n<think>\n"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=450, temperature=0.2, do_sample=True)
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return "<think>\n" + response

demo = gr.ChatInterface(fn=solve, title="⚙️ Vigyan-1.5B Engineering: Electrical, Mechanical & Systems", description="DeepSeek-R1 Distill Qwen 1.5B Architecture | Created by Shreyansh Singh", examples=["Calculate resonant frequency and quality factor (Q) of a parallel RLC circuit with R = 100 ohms, L = 10 mH, C = 1 uF.", "Derive the maximum bending moment and deflection of a cantilever beam with length L subject to point load P at free end.", "Explain difference between P, PI, and PID control loops and how derivative action prevents overshoot in closed loops."])
demo.launch(share=True)